# 异常检测

在本练习中，你将实现异常检测算法，并将其应用于检测网络中出现故障的服务器。



# 大纲
- [ 1 - 软件包 ](#1)
- [ 2 - 异常检测](#2)
  - [ 2.1 问题陈述](#2.1)
  - [ 2.2 数据集](#2.2)
  - [ 2.3 高斯分布](#2.3)
    - [ 练习 1](#ex01)
    - [ 练习 2](#ex02)
  - [ 2.4 高维数据集](#2.4)

<a name="1"></a>
## 1 - 软件包

首先，运行下面的单元格，导入完成本作业所需的全部软件包。
- [NumPy](www.numpy.org) 是在 Python 中处理矩阵的基础软件包。
- [matplotlib](http://matplotlib.org) 是 Python 中著名的绘图库。
- ``utils.py`` 包含本作业的辅助函数。你无需修改该文件中的代码。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *

%matplotlib inline

<a name="2"></a>
## 2 - 异常检测

<a name="2.1"></a>
### 2.1 问题陈述

在本练习中，你将实现一个异常检测算法，用于检测服务器计算机的异常行为。

该数据集包含两个特征：
   * 各服务器响应的吞吐量 (mb/s)，以及
   * 延迟 (ms)。

在服务器运行期间，你收集了 $m=307$ 个关于其行为的样本，因而得到一个无标签数据集 $\{x^{(1)}, \ldots, x^{(m)}\}$。
* 你怀疑其中绝大多数样本都是服务器正常运行时的“正常”（非异常）样本，但该数据集中也可能包含一些服务器行为异常的样本。

你将使用高斯模型检测数据集中的异常样本。
* 首先，你将从一个二维数据集开始，从而能够将算法的运行过程可视化。
* 在该数据集上拟合高斯分布，然后找出概率非常低、因而可视为异常的值。
* 之后，将异常检测算法应用于一个包含多个维度的更大数据集。

<a name="2.2"></a>
### 2.2 数据集

首先，你将加载此任务的数据集。
- 下面所示的 `load_data()` 函数会将数据加载到变量 `X_train`、`X_val` 和 `y_val` 中
    - 你将使用 `X_train` 拟合高斯分布
    - 你将使用 `X_val` 和 `y_val` 作为交叉验证集，以选择阈值并区分异常样本与正常样本

In [ ]:
# Load the dataset
X_train, X_val, y_val = load_data()

#### 查看变量
让我们进一步熟悉数据集。  
- 一个很好的起点是直接打印每个变量，看看其中包含什么。

下面的代码会打印每个变量的前五个元素

In [ ]:
# Display the first five elements of X_train
print("The first 5 elements of X_train are:\n", X_train[:5])  

In [ ]:
# Display the first five elements of X_val
print("The first 5 elements of X_val are\n", X_val[:5])  

In [ ]:
# Display the first five elements of y_val
print("The first 5 elements of y_val are\n", y_val[:5])  

#### 检查变量的维度

查看数据维度是熟悉数据的另一种实用方法。

下面的代码会打印 `X_train`、`X_val` 和 `y_val` 的形状。

In [ ]:
print ('The shape of X_train is:', X_train.shape)
print ('The shape of X_val is:', X_val.shape)
print ('The shape of y_val is: ', y_val.shape)

#### 可视化数据

开始任何任务之前，通过可视化来理解数据通常很有帮助。
- 对于此数据集，由于只有两个可绘制属性（吞吐量和延迟），可以使用散点图（`X_train`）进行可视化

- 绘制的图应与下图类似
<img src="images/figure1.png" width="500" height="500">

In [ ]:
# Create a scatter plot of the data. To change the markers to blue "x",
# we used the 'marker' and 'c' parameters
plt.scatter(X_train[:, 0], X_train[:, 1], marker='x', c='b') 

# Set the title
plt.title("The first dataset")
# Set the y-axis label
plt.ylabel('Throughput (mb/s)')
# Set the x-axis label
plt.xlabel('Latency (ms)')
# Set axis range
plt.axis([0, 30, 0, 30])
plt.show()

<a name="2.3"></a>
### 2.3 高斯分布

要进行异常检测，首先需要为数据的分布拟合一个模型。

* 给定训练集 $\{x^{(1)}, ..., x^{(m)}\}$，你需要估计每个特征 $x_i$ 的高斯分布。

* 回顾一下，高斯分布定义为

   $$ p(x ; \mu,\sigma ^2) = \frac{1}{\sqrt{2 \pi \sigma ^2}}\exp^{ - \frac{(x - \mu)^2}{2 \sigma ^2} }$$

   其中，$\mu$ 是均值，$\sigma^2$ 控制方差。
   
* 对于每个特征 $i = 1\ldots n$，你需要找到参数 $\mu_i$ 和 $\sigma_i^2$，以拟合第 $i$ 个维度 $\{x_i^{(1)}, ..., x_i^{(m)}\}$（每个样本的第 $i$ 个维度）中的数据。

### 2.2.1 估计高斯分布的参数

**实现**：

你的任务是补全下面 `estimate_gaussian` 中的代码。

<a name="ex01"></a>
### 练习 1

请完成下面的 `estimate_gaussian` 函数，以计算 `mu`（`X` 中每个特征的均值）和 `var`（`X` 中每个特征的方差）。

你可以使用以下方程估计第 $i$ 个特征的参数（$\mu_i$、$\sigma_i^2$）。要估计均值，可以使用：

$$\mu_i = \frac{1}{m} \sum_{j=1}^m x_i^{(j)}$$

要估计方差，可以使用：
$$\sigma_i^2 = \frac{1}{m} \sum_{j=1}^m (x_i^{(j)} - \mu_i)^2$$

如果遇到困难，可以查看下方单元格之后给出的提示，以帮助你完成实现。

In [ ]:
# UNQ_C1
# GRADED FUNCTION: estimate_gaussian

def estimate_gaussian(X): 
    """
    Calculates mean and variance of all features 
    in the dataset
    
    Args:
        X (ndarray): (m, n) Data matrix
    
    Returns:
        mu (ndarray): (n,) Mean of all features
        var (ndarray): (n,) Variance of all features
    """

    m, n = X.shape
    
    ### START CODE HERE ### 
    
    ### END CODE HERE ### 
        
    return mu, var

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
  
   * 你可以用两种方式实现此函数：
      * 1 - 使用两个嵌套的 for 循环——一个循环遍历 `X` 的**列**（每个特征），另一个循环遍历各数据点。
      * 2 - 采用向量化方式，使用带 `axis = 0` 参数的 `np.sum()`（因为我们希望对每列求和）

    
   * 对于向量化实现，可以按如下方式组织此函数的整体实现：
     ```python  
    def estimate_gaussian(X): 
        m, n = X.shape
    
        ### START CODE HERE ### 
        mu = # Your code here to calculate the mean of every feature
        var = # Your code here to calculate the variance of every feature 
        ### END CODE HERE ### 
        
        return mu, var
    ```

    如果仍然没有思路，可以查看下面给出的提示，了解如何计算 `mu` 和 `var`。
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 mu 的提示</b></font></summary>
           &emsp; &emsp; 可以使用带 `axis = 0` 参数的 <a href="https://numpy.org/doc/stable/reference/generated/numpy.sum.html">np.sum</a>，得到数组每一列的总和
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 mu 的更多提示</b></font></summary>
               &emsp; &emsp; 可以按如下方式计算 mu：<code>mu = 1 / m * np.sum(X, axis = 0)</code>
           </details>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 var 的提示</b></font></summary>
           &emsp; &emsp; 可以使用带 `axis = 0` 参数的 <a href="https://numpy.org/doc/stable/reference/generated/numpy.sum.html">np.sum</a>，得到数组每一列的总和，并使用 <code>**2</code> 取平方。
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 var 的更多提示</b></font></summary>
               &emsp; &emsp; 可以按如下方式计算 var：<code> var = 1 / m * np.sum((X - mu) ** 2, axis = 0)</code>
           </details>
    </details>
    
</details>

你可以运行以下测试代码，检查实现是否正确：

In [ ]:
# Estimate mean and variance of each feature
mu, var = estimate_gaussian(X_train)              

print("Mean of each feature:", mu)
print("Variance of each feature:", var)
    
# UNIT TEST
from public_tests import *
estimate_gaussian_test(estimate_gaussian)

**预期输出**：
<table>
  <tr>
    <td> <b>各特征的均值：<b>  </td>
    <td> [14.11222578 14.99771051]</td>
   </tr>
   <tr>
    <td> <b>各特征的方差：<b>  </td>
     <td> [1.83263141 1.70974533] </td>
  </tr>
</table>

现在，你已经完成 `estimate_gaussian` 中的代码，我们将可视化拟合后的高斯分布等高线。

你应会得到一张类似下图的图。
<img src="images/figure2.png" width="500" height="500">


从图中可以看到，大多数样本位于概率最高的区域，而异常样本位于概率较低的区域。

In [ ]:
# Returns the density of the multivariate normal
# at each data point (row) of X_train
p = multivariate_gaussian(X_train, mu, var)

#Plotting code 
visualize_fit(X_train, mu, var)

### 2.2.2 选择阈值 $\epsilon$

现在您已经估计了高斯分布参数，可以研究在该分布下哪些样本的概率非常高、哪些样本的概率非常低。

* 低概率样本更有可能是数据集中的异常点。
* 判断哪些样本是异常点的一种方法，是基于交叉验证集选择阈值。

在本节中，您将完成 `select_threshold` 中的代码，使用交叉验证集上的 $F_1$ 分数选择阈值 $\varepsilon$。

* 为此，我们将使用交叉验证集 $\{(x_{\rm cv}^{(1)}, y_{\rm cv}^{(1)}),\ldots, (x_{\rm cv}^{(m_{\rm cv})}, y_{\rm cv}^{(m_{\rm cv})})\}$，其中标签 $y=1$ 对应异常样本，$y=0$ 对应正常样本。
* 对每个交叉验证样本，我们将计算 $p(x_{\rm cv}^{(i)})$。由这些概率组成的向量 $p(x_{\rm cv}^{(1)}), \ldots, p(x_{\rm cv}^{(m_{\rm cv)}})$ 将以向量 `p_val` 的形式传入 `select_threshold`。
* 对应的标签 $y_{\rm cv}^{(1)}, \ldots, y_{\rm cv}^{(m_{\rm cv)}}$ 将以向量 `y_val` 的形式传入同一个函数。

<a name="ex02"></a>
### 练习 2
请补全下面的 `select_threshold` 函数，根据验证集（`p_val`）的结果和真实标签（`y_val`），找到用于选择离群点的最佳阈值。

* 在所提供的代码 `select_threshold` 中，已有一个循环会尝试 $\varepsilon$ 的许多不同取值，并根据 $F_1$ 分数选择最佳 $\varepsilon$。

* 你需要实现代码，计算选择 `epsilon` 作为阈值时的 F1 分数，并将该值存入 `F1`。

  * 回顾一下，如果一个样本 $x$ 的概率 $p(x) < \varepsilon$ 很低，就会被分类为异常。
        
  * 然后，可以按如下方式计算精确率和召回率：
   $$\begin{aligned}
   prec&=&\frac{tp}{tp+fp}\\
   rec&=&\frac{tp}{tp+fn},
   \end{aligned}$$，其中
    * $tp$ 是真正例数量：真实标签表明它是异常，且我们的算法正确地将其分类为异常。
    * $fp$ 是假正例数量：真实标签表明它不是异常，但我们的算法错误地将其分类为异常。
    * $fn$ 是假负例数量：真实标签表明它是异常，但我们的算法错误地将其分类为非异常。

  * $F_1$ 分数使用精确率（$prec$）和召回率（$rec$）按下式计算：
    $$F_1 = \frac{2\cdot prec \cdot rec}{prec + rec}$$

**实现说明：**
为了计算 $tp$、$fp$ 和 $fn$，你可以采用向量化实现，而不必遍历所有样本。


如果遇到困难，可以查看下方单元格之后给出的提示，以帮助你完成实现。

In [ ]:
# UNQ_C2
# GRADED FUNCTION: select_threshold

def select_threshold(y_val, p_val): 
    """
    Finds the best threshold to use for selecting outliers 
    based on the results from a validation set (p_val) 
    and the ground truth (y_val)
    
    Args:
        y_val (ndarray): Ground truth on validation set
        p_val (ndarray): Results on validation set
        
    Returns:
        epsilon (float): Threshold chosen 
        F1 (float):      F1 score by choosing epsilon as threshold
    """ 

    best_epsilon = 0
    best_F1 = 0
    F1 = 0
    
    step_size = (max(p_val) - min(p_val)) / 1000
    
    for epsilon in np.arange(min(p_val), max(p_val), step_size):
    
        ### START CODE HERE ### 
        
        ### END CODE HERE ### 
        
        if F1 > best_F1:
            best_F1 = F1
            best_epsilon = epsilon
        
    return best_epsilon, best_F1

<details>
  <summary><font size="3" color="darkgreen"><b>单击查看提示</b></font></summary>

   * 以下是向量化实现中该函数整体实现的组织方式：
     ```python  
    def select_threshold(y_val, p_val): 
        best_epsilon = 0
        best_F1 = 0
        F1 = 0
    
        step_size = (max(p_val) - min(p_val)) / 1000
    
        for epsilon in np.arange(min(p_val), max(p_val), step_size):
    
            ### START CODE HERE ### 
            predictions = # Your code here to calculate predictions for each example using epsilon as threshold
        
            tp = # Your code here to calculate number of true positives
            fp = # Your code here to calculate number of false positives
            fn = # Your code here to calculate number of false negatives
        
            prec = # Your code here to calculate precision
            rec = # Your code here to calculate recall
        
            F1 = # Your code here to calculate F1
            ### END CODE HERE ### 
        
            if F1 > best_F1:
                best_F1 = F1
                best_epsilon = epsilon
        
        return best_epsilon, best_F1
    ```

    如果仍然遇到困难，可以查看下方提示，了解如何计算各个变量。
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算预测的提示</b></font></summary>
           &emsp; &emsp; 如果样本 𝑥 的概率 $p(x) < \epsilon$ 很低，则将其分类为异常。要获得每个样本的预测（0/False 表示正常，1/True 表示异常），可以使用 <code>predictions = (p_val < epsilon)</code>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 tp、fp、fn 的提示</b></font></summary>
           &emsp; &emsp; 
        <ul>
          <li>如果在一个 $n$ 维二值向量中有多个二值，可以使用以下方式找出该向量中有多少个值为 0：`np.sum(v == 0)`</li>
          <li>还可以对这种二值向量应用逻辑 *and* 运算符。例如，`predictions` 是一个大小等于交叉验证集样本数的二值向量；如果算法认为 $x_{\rm cv}^{(i)}$ 是异常，则第 $i$ 个元素为 1，否则为 0。</li>
          <li>然后，例如，可以按如下方式计算假阳性的数量：  
<code>fp = sum((predictions == 1) & (y_val == 0))</code>。</li>
        </ul>
         <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 tp、fn 的更多提示</b></font></summary>
               &emsp; &emsp;
             <ul>
              <li>可以按如下方式计算 tp：<code> tp = np.sum((predictions == 1) & (y_val == 1))</code></li>
              <li>可以按如下方式计算 tn：<code> fn = np.sum((predictions == 0) & (y_val == 1))</code></li>  
              </ul>
          </details>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>计算精确率的提示</b></font></summary>
           &emsp; &emsp; 可以按如下方式计算精确率：<code>prec = tp / (tp + fp)</code>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>计算召回率的提示</b></font></summary>
           &emsp; &emsp; 可以按如下方式计算召回率：<code>rec = tp / (tp + fn)</code>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>计算 F1 的提示</b></font></summary>
           &emsp; &emsp; 可以按如下方式计算 F1：<code>F1 = 2 * prec * rec / (prec + rec)</code>
    </details>
    
</details>

你可以使用下面的代码检查实现

In [ ]:
p_val = multivariate_gaussian(X_val, mu, var)
epsilon, F1 = select_threshold(y_val, p_val)

print('Best epsilon found using cross-validation: %e' % epsilon)
print('Best F1 on Cross Validation Set: %f' % F1)
    
# UNIT TEST
select_threshold_test(select_threshold)


**预期输出**：
<table>
  <tr>
    <td> <b>使用交叉验证找到的最佳 epsilon：<b>  </td>
    <td> 8.99e-05</td> 
   </tr>    
   <tr>
    <td> <b>交叉验证集上的最佳 F1：<b>  </td>
     <td> 0.875 </td> 
  </tr>
</table>

现在，我们将运行你的异常检测代码，并在图中圈出异常点（见下方图 3）。

<img src="images/figure3.png" width="500" height="500">

In [ ]:
# Find the outliers in the training set 
outliers = p < epsilon

# Visualize the fit
visualize_fit(X_train, mu, var)

# Draw a red circle around those outliers
plt.plot(X_train[outliers, 0], X_train[outliers, 1], 'ro',
         markersize= 10,markerfacecolor='none', markeredgewidth=2)

<a name="2.4"></a>
### 2.4 高维数据集

现在，我们将在一个更真实、难度也大得多的数据集上运行你实现的异常检测算法。

在此数据集中，每个样本由 11 个特征描述，能够捕捉计算服务器的更多属性。

首先，加载数据集。

- 下面所示的 `load_data()` 函数将数据加载到变量 `X_train_high`、`X_val_high` 和 `y_val_high` 中
    - 使用 `_high` 是为了将这些变量与上一部分使用的变量区分开来
    - 我们将使用 `X_train_high` 拟合高斯分布
    - 我们将使用 `X_val_high` 和 `y_val_high` 作为交叉验证集，以选择阈值并判断样本是异常还是正常

In [ ]:
# load the dataset
X_train_high, X_val_high, y_val_high = load_data_multi()

#### 检查变量的维度

让我们检查这些新变量的维度，以便熟悉数据

In [ ]:
print ('The shape of X_train_high is:', X_train_high.shape)
print ('The shape of X_val_high is:', X_val_high.shape)
print ('The shape of y_val_high is: ', y_val_high.shape)

#### 异常检测

现在，让我们在这个新数据集上运行异常检测算法。

下面的代码将使用你的代码来：
* 估计高斯参数（$\mu_i$ 和 $\sigma_i^2$）
* 计算训练数据 `X_train_high`（用于估计高斯参数的数据）和交叉验证集 `X_val_high` 的概率。
* 最后，使用 `select_threshold` 找到最佳阈值 $\varepsilon$。

In [ ]:
# Apply the same steps to the larger dataset

# Estimate the Gaussian parameters
mu_high, var_high = estimate_gaussian(X_train_high)

# Evaluate the probabilites for the training set
p_high = multivariate_gaussian(X_train_high, mu_high, var_high)

# Evaluate the probabilites for the cross validation set
p_val_high = multivariate_gaussian(X_val_high, mu_high, var_high)

# Find the best threshold
epsilon_high, F1_high = select_threshold(y_val_high, p_val_high)

print('Best epsilon found using cross-validation: %e'% epsilon_high)
print('Best F1 on Cross Validation Set:  %f'% F1_high)
print('# Anomalies found: %d'% sum(p_high < epsilon_high))

**预期输出**：
<table>
  <tr>
    <td> <b>使用交叉验证找到的最佳 epsilon：<b>  </td>
    <td> 1.38e-18</td>
   </tr>
   <tr>
    <td> <b>交叉验证集上的最佳 F1：<b>  </td>
     <td> 0.615385 </td>
  </tr>
    <tr>
    <td> <b>发现的异常数量：<b>  </td>
     <td>  117 </td>
  </tr>
</table>